In [45]:
import pandas as pd

link_stream = pd.read_csv('data/CollegeMsg.txt', delimiter=' ', names=['source', 'destination', 'timestamp'])

In [49]:
import pandas as pd


nodes = pd.concat([link_stream['source'], link_stream['destination']]).unique()

node2id = {node: idx for idx, node in enumerate(nodes)}

link_stream['source'] = link_stream['source'].map(node2id)
link_stream['destination'] = link_stream['destination'].map(node2id)
link_stream['idx'] = range(len(link_stream))

In [39]:
link_stream.to_csv('data/CollegeMsg.csv', index=False)

In [3]:
import pandas as pd

link_stream = pd.read_csv('data/CollegeMsg.csv')

In [4]:
import torch
import pandas as pd
import numpy as np
from typing import Dict, Tuple


class OfflineStateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0
        self.node_time_pos = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.node_time_indptr = None   # shape: [num_nodes + 1], int64
        self.node_time_values = None   # shape: [total_appearances], int32/int64 (timestamps)

        self.A_base = None 
        self.S_base = None 
        self.p_prev = None 

    def pre_scan_data(self, link_stream):
        self.m = float(len(link_stream))
        print(f"Total links: {int(self.m)}")

        all_nodes = pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        t_max = link_stream['timestamp'].max()
        t_min = link_stream['timestamp'].min()
        total_duration = t_max - t_min
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"T_duration = {total_duration}")
    
        # compute node lifespans
        sources = link_stream[['source', 'timestamp']].rename(columns={'source': 'node'})
        destinations = link_stream[['destination', 'timestamp']].rename(columns={'destination': 'node'})
        all_events = pd.concat([sources, destinations], ignore_index=True)
        
        node_stats = all_events.groupby('node')['timestamp'].agg(['min', 'max'])
        
        epsilon = 1.0 
        lifespans = (node_stats['max'] - node_stats['min']).clip(lower=epsilon)
        
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        
        self.node_lifespans.fill_(epsilon)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")

        # Sort by (node, timestamp). Use stable sort to keep deterministic order for same timestamps.
        all_events_sorted = all_events.sort_values(['node', 'timestamp'], kind='mergesort')
        nodes_np = all_events_sorted['node'].to_numpy(dtype=np.int64, copy=False)
        times_np = all_events_sorted['timestamp'].to_numpy(copy=False)


        if np.issubdtype(times_np.dtype, np.integer):
            # choose int32 if safe
            if times_np.min() >= np.iinfo(np.int32).min and times_np.max() <= np.iinfo(np.int32).max:
                times_np = times_np.astype(np.int32, copy=False)
            else:
                times_np = times_np.astype(np.int64, copy=False)
        else:
            # float timestamps: keep float32/float64; typically float32 is enough
            times_np = times_np.astype(np.float32, copy=False)

        # Count appearances per node
        counts = np.bincount(nodes_np, minlength=self.num_nodes).astype(np.int64)  # length n
        indptr = np.empty(self.num_nodes + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(counts, out=indptr[1:])

        # Save to torch (CPU/GPU)
        self.node_time_indptr = torch.from_numpy(indptr).to(self.device)
        self.node_time_values = torch.from_numpy(times_np).to(self.device)

        print(f"Total node appearances stored: {self.node_time_values.numel()} (expected ~ {2*int(self.m)})")

        self.reset_dynamic_states()

    def reset_time_pos(self):
        self.node_time_pos.zero_()

    def reset_dynamic_states(self):
        """
        Initialize:
          A_base[u,c] = T_u / K  (uniform over communities across u's lifespan)
          p_prev[u,c] = 1/K
          S_base[c]   = sum_u k_u * sqrt( (A_base[u,c]/T_u) ) = sum_u k_u * sqrt(1/K)
        """
        N, K = self.num_nodes, self.num_communities
        Tu = self.node_lifespans.clamp_min(1)  # [N]

        # A_base: [N,K]
        self.A_base = (Tu[:, None] / float(K)).expand(N,K).to(device=self.device, dtype=torch.float32).clone()

        # p_prev: [N,K]
        self.p_prev = torch.full((N, K), 1.0 / float(K), device=self.device, dtype=torch.float32)

        # S_base: [K]
        ratio = (self.A_base / Tu[:, None]).clamp_min(1)  # should be 1/K initially
        self.S_base = (self.global_degree.to(self.device, dtype=torch.float32)[:, None] * torch.sqrt(ratio)).sum(dim=0)


    def get_node_times(self, u: int) -> torch.Tensor:
        """Return 1D tensor of all appearance timestamps of node u (sorted)."""
        start = int(self.node_time_indptr[u].item())
        end = int(self.node_time_indptr[u + 1].item())
        return self.node_time_values[start:end]

    def get_next_time(self, u: int, j: int):
        """Return (t_j, t_{j+1}) for node u's j-th appearance. Raises if j+1 out of range."""
        start = int(self.node_time_indptr[u].item())
        end = int(self.node_time_indptr[u + 1].item())
        if start + j + 1 >= end:
            return None, None
        t_j = self.node_time_values[start + j]
        t_next = self.node_time_values[start + j + 1]
        return t_j, t_next
    
    def get_current_interval(self, u: int):
        """
        O(1) 返回当前出现时间 t1、下一次出现时间 t2，以及 delta = t2 - t1
        如果这是最后一次出现，返回 delta=0
        """
        u = int(u)
        pos = int(self.node_time_pos[u].item())

        start = int(self.node_time_indptr[u].item())
        end   = int(self.node_time_indptr[u + 1].item())

        idx = start + pos

        # Safety check
        if idx >= end:
            # Should not normally happen
            return None, None, 0.0

        t1 = self.node_time_values[idx]

        if idx + 1 < end:
            t2 = self.node_time_values[idx + 1]
            delta = float(t2 - t1)
        else:
            # last appearance: no future interval
            t2 = None
            delta = 0.0

        return t1, t2, delta
    
    def advance_time_pos(self, u: int):
        self.node_time_pos[u] += 1

    def get_delta_for_batch(
        self,
        src: torch.Tensor,  # [B] long
        dst: torch.Tensor,  # [B] long
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[int, int]]:
        """
        Return delta_src/delta_dst for each occurrence in this batch.
        Handles multiple appearances within the batch by using a temporary pos per node.

        Also returns occ_counts: {u: count_in_this_batch} for committing node_time_pos.
        """
        assert src.dtype == torch.long and dst.dtype == torch.long
        B = src.numel()
        device = self.device

        # keep deltas on device, float32
        delta_src = torch.zeros(B, device=device, dtype=torch.float32)
        delta_dst = torch.zeros(B, device=device, dtype=torch.float32)

        # temp positions for nodes touched in this batch
        temp_pos: Dict[int, int] = {}
        occ_counts: Dict[int, int] = {}

        def _next_delta(u: int) -> float:
            # get current pos for u (global + temp offset)
            base_pos = int(self.node_time_pos[u].item())
            offset = temp_pos.get(u, 0)
            pos = base_pos + offset

            start = int(self.node_time_indptr[u].item())
            end = int(self.node_time_indptr[u + 1].item())
            idx = start + pos

            # advance temp pos for next time
            temp_pos[u] = offset + 1
            occ_counts[u] = occ_counts.get(u, 0) + 1

            # if out of range or last appearance -> delta 0
            if idx + 1 >= end:
                return 0.0

            t1 = self.node_time_values[idx]
            t2 = self.node_time_values[idx + 1]

            # Convert to float on CPU/GPU safely (t may be int/float tensor)
            # delta is a scalar used as weight only
            return float((t2 - t1).item())

        # process in event order: src then dst (consistent with CSC sequential ordering too)
        for i in range(B):
            u = int(src[i].item())
            v = int(dst[i].item())
            delta_src[i] = _next_delta(u)
            delta_dst[i] = _next_delta(v)

        return delta_src, delta_dst, occ_counts


    @torch.no_grad()
    def commit_batch(
        self,
        delta_a_nodes: Dict[int, torch.Tensor],  # {u: [K]} (grad tensor, we detach outside or here)
        last_p_nodes: Dict[int, torch.Tensor],   # {u: [K]} last p in this batch
        occ_counts: Dict[int, int],              # {u: count}
    ):
        """
        Update buffers:
          A_base, S_base, p_prev, node_time_pos
        Must be called AFTER optimizer.step() and under no_grad.
        """
        #eps = self.eps
        eps = 1.0
        for u, dA in delta_a_nodes.items():
            u = int(u)
            dA = dA.detach()

            ku = self.global_degree[u]
            Tu = self.node_lifespans[u].clamp_min(eps)

            # old sqrt ratio
            old_ratio = (self.A_base[u] / Tu).clamp_min(eps)
            old_sqrt = torch.sqrt(old_ratio)

            # update A_base
            self.A_base[u] += dA

            # new sqrt ratio
            new_ratio = (self.A_base[u] / Tu).clamp_min(eps)
            new_sqrt = torch.sqrt(new_ratio)

            # update S_base (vector over communities)
            self.S_base += ku * (new_sqrt - old_sqrt)

        # update p_prev (for CSC)
        for u, p_last in last_p_nodes.items():
            self.p_prev[int(u)] = p_last.detach()

        # commit node_time_pos (advance by how many times node appeared in this batch)
        for u, cnt in occ_counts.items():
            self.node_time_pos[int(u)] += int(cnt)

In [171]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 5
state_mgr = OfflineStateManager(num_nodes, num_comms, device=device)
state_mgr.pre_scan_data(link_stream)


num_nodes = 1899 
Total links: 59835
T_duration = 16736181
Max Lifespan: 16625344.0
Total node appearances stored: 119670 (expected ~ 119670)


In [172]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.01
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'mps'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [173]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'CollegeMsg'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/CollegeMsg.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/CollegeMsg_node.npy), use zero vector(dim=16)...
The dataset has 59835 interactions, involving 1899 different nodes


In [174]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [175]:
from tgn.utils.my_data import get_data, compute_time_statistics
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= \
  compute_time_statistics(
      full_data.sources, 
      full_data.destinations, 
      full_data.timestamps
  )


In [176]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.01
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16


device = 'cuda' if torch.cuda.is_available() else 'mps'
aggregator = 'last'
memory_update_at_end = False
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'college_msg'

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,

    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 

).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [177]:
import math 
import logging
import time


BATCH_SIZE = 200

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 300


In [182]:
from tgn.model.loss import longitudinal_modularity_loss
importlib.reload(sys.modules['tgn.model.loss'])
NUM_EPOCHS = 3


for epoch in range(NUM_EPOCHS):
    tgn = tgn.train()

    start_epoch_time = time.time()

    if USE_MEMORY:
        tgn.memory.__init_memory__()
    state_mgr.reset_dynamic_states()
    state_mgr.reset_time_pos()
    tgn.set_neighbor_finder(ngh_finder)
    
    epoch_loss = 0.0
    epoch_obs_loss = 0.0
    epoch_null_loss = 0.0
    epoch_csc_loss = 0.0
    epoch_balance_loss = 0.0
    epoch_conf_loss = 0.0
    epoch_steps = 0

    logger.info(f'Starting epoch {epoch+1} / {NUM_EPOCHS} ')
    for k in range(0, num_batches):
        
        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]
        edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
        
        src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

        delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)
        p_src, p_dst = tgn.compute_community_prob(sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch) # [B, K]
        
        # Compute longitudinal modularity loss
        loss, last_p_nodes, last_components, delta_a_nodes, terms_raw, extra = longitudinal_modularity_loss(
            p_src, p_dst,
            src, dst,
            delta_src, delta_dst,
            state_mgr.A_base, state_mgr.S_base, state_mgr.global_degree,
            state_mgr.m, state_mgr.total_duration,
            state_mgr.p_prev,
            obs_weight=0.0,
            null_weight=1.0,
            csc_weight=0.0,
            collapse_weight=0.0,
            conf_weight=0.0,
            csc_norm="l2"
        )
        # ===== debug prints =====
        DeltaS_norm = extra["DeltaS"].norm().item()
        delta_energy = extra["delta_energy"].item()

        # A0 touched mean: 只统计本 batch 触及的节点的 A_base[u,:] 的均值
        with torch.no_grad():
            A0 = state_mgr.A_base.detach()
            touched = list(delta_a_nodes.keys())
            if len(touched) > 0:
                idx = torch.tensor(touched, device=A0.device, dtype=torch.long)
                A0_touched_mean = A0.index_select(0, idx).mean().item()
                A0_touched_max  = A0.index_select(0, idx).max().item()
            else:
                A0_touched_mean = 0.0
                A0_touched_max  = 0.0

        print(f"[batch {k+1}] |DeltaS|={DeltaS_norm:.3e}  delta_energy={delta_energy:.3e}  "
            f"A0_touched_mean={A0_touched_mean:.3e}  A0_touched_max={A0_touched_max:.3e}  "
            f"touched_nodes={len(touched)}")
        # ===== end debug =====
        
        def grad_stat(loss_term, params, name):
            grads = torch.autograd.grad(
                loss_term, params,
                retain_graph=True,
                allow_unused=True
            )
            grads = [g for g in grads if g is not None]
            if len(grads) == 0:
                print(f"{name}: grad is None (no path)")
                return
            mean_abs = torch.stack([g.abs().mean() for g in grads]).mean().item()
            max_abs  = torch.stack([g.abs().max()  for g in grads]).max().item()
            l2_norm  = torch.sqrt(torch.stack([(g*g).sum() for g in grads]).sum()).item()
            print(f"{name}: mean|g|={mean_abs:.3e}, max|g|={max_abs:.3e}, ||g||2={l2_norm:.3e}")
        
        if k<3:
            print(f'{k+1} batch: ')
            params = list(tgn.community_projector.parameters())
            grad_stat(terms_raw["obs"],  params, "obs")
            grad_stat(terms_raw["null"], params, "null")
            grad_stat(terms_raw["csc"],  params, "csc")
        '''
        print("p_src requires_grad:", p_src.requires_grad)
        print("p_dst requires_grad:", p_dst.requires_grad)
        print("delta_src mean/max:", float(delta_src.mean()), float(delta_src.max()))
        print("delta_dst mean/max:", float(delta_dst.mean()), float(delta_dst.max()))
        print("delta_src zero ratio:", float((delta_src==0).float().mean()))
        print("delta_dst zero ratio:", float((delta_dst==0).float().mean()))
        print("delta_a_nodes size:", len(delta_a_nodes))
        '''

        '''
        # 只检查 null loss
        loss_null = terms_raw  # 如果你现在 loss 就是 null；否则取 loss_null 变量

        # 取一个 batch 里的节点 id（比如第 0 个 src）
        i = 0
        u = int(src[i].item())

        # 计算 dL/dp_src[i]，这是一个 K 维向量
        g_p = torch.autograd.grad(loss, p_src, retain_graph=True)[0]  # [B,K]
        g_u = g_p[i].detach()

        print("u =", u)
        print("dL/dp (K-dim) stats:",
            "mean", g_u.mean().item(),
            "std", g_u.std(unbiased=False).item(),
            "min", g_u.min().item(),
            "max", g_u.max().item())
        print("dL/dp values (first 10):", g_u[:10].cpu().numpy())
        '''
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        

        state_mgr.commit_batch(delta_a_nodes, last_p_nodes, occ_counts)
        if USE_MEMORY:
            tgn.memory.detach_memory()
        
        epoch_loss += float(loss.detach().item())
        epoch_obs_loss  += float(last_components[0].item())
        epoch_null_loss += float(last_components[1].item())
        epoch_csc_loss  += float(last_components[2].item())
        epoch_balance_loss += float(last_components[3].item())
        epoch_conf_loss += float(last_components[4].item())
        epoch_steps += 1
        #print(f'Epoch {epoch} Batch {k+1}/{num_batches} null model loss: {epoch_null_loss / epoch_steps} ')    
    if epoch_steps > 0:
        mean_loss = epoch_loss / epoch_steps
        mean_obs  = epoch_obs_loss  / epoch_steps
        mean_null = epoch_null_loss / epoch_steps
        mean_csc  = epoch_csc_loss  / epoch_steps
        mean_balance = epoch_balance_loss / epoch_steps
        mean_conf = epoch_conf_loss / epoch_steps
    
        logger.info(
            f"[Epoch {epoch}] mean loss={mean_loss:.6f} | "
            f"obs={mean_obs:.6f}, null={mean_null:.6f}, csc={mean_csc:.6f}, balance={mean_balance:.6f}, conf={mean_conf:.6f} "
        )

    if USE_MEMORY:
        # Backup and restore memory
        memory_backup = tgn.memory.backup_memory()
        tgn.memory.restore_memory(memory_backup)

INFO:root:Starting epoch 1 / 3 


ValueError: not enough values to unpack (expected 6, got 5)

In [ ]:
import csv
import torch
from tqdm import tqdm

@torch.no_grad()
def export_event_communities(
    tgn,
    full_data,
    device,
    output_path,
    batch_size=200
):
    """
    导出：
    source, destination, timestamp,
    source_assignment, destination_assignment,
    source_comm, destination_comm
    """
    tgn.eval()

    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)

        # header
        writer.writerow([
            "source",
            "destination",
            "timestamp",
            "source_assignment",
            "destination_assignment",
            "source_comm",
            "destination_comm"
        ])

        num_instance = len(full_data.sources)
        num_batches = (num_instance + batch_size - 1) // batch_size

        for k in tqdm(range(num_batches), desc="Exporting events"):
            start_idx = k * batch_size
            end_idx   = min(num_instance, (k + 1) * batch_size)

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
            p_src, p_dst = tgn.compute_community_prob(sources_batch, 
                                                      destinations_batch, 
                                                      timestamps_batch, 
                                                      edge_idxs_batch) # [B, K]


            # hard labels
            c_src = p_src.argmax(dim=1)
            c_dst = p_dst.argmax(dim=1)

            # write rows
            print(sources_batch)
            for i in range(sources_batch.size(0)):
                writer.writerow([
                    int(sources_batch[i].item()),
                    int(destinations_batch[i].item()),
                    float(timestamps_batch[i].item()),
                    p_src[i].cpu().tolist(),   # soft assignment
                    p_dst[i].cpu().tolist(),
                    int(c_src[i].item()),      # hard label
                    int(c_dst[i].item())
                ])

output_csv = "event_community_assignments.csv"

export_event_communities(
    tgn=tgn,
    full_data=full_data,
    device='mps',
    output_path=output_csv,
    batch_size=BATCH_SIZE
)

print("Saved to:", output_csv)

Exporting events:   0%|          | 0/300 [00:00<?, ?it/s]


AssertionError: Trying to update memory to time in the past